In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.ops import unary_union

from pathlib import Path
from tqdm.auto import tqdm

datasets = Path("/nas/cee-ice/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/owen_reservoirs/Columbia")
metadata_dir = save_dir / "metadata"

subbasins = gpd.read_parquet(metadata_dir / 'subbasins_4km2.parquet')
subbasins.index = subbasins.index.astype(str)
subbasins.geometry = subbasins.geometry.centroid
footprint_geom = unary_union(subbasins.geometry.envelope)

/tmp/ipykernel_3708119/2484409471.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subbasins.geometry = subbasins.geometry.centroid


In [3]:
# Read in hydroatlas
gdb_path = datasets / "HydroSHEDS" / "HydroATLAS" / "BasinATLAS_v10.gdb"
layer_name = "BasinATLAS_v10_lev09"

hydroatlas = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio", use_arrow=True)
hydroatlas.replace({'wet_cl_smj':-9999}, 13, inplace=True)
hydroatlas = hydroatlas.set_index('HYBAS_ID')

# hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
# hydroatlas_filt

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pyogrio/raw.py:337: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  table = reader.read_all()


In [5]:
# hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
# hydroatlas_filt

In [14]:
hybas_pairs.index

Index(['23021321', '23021261', '23021244', '23021122', '23021112', '23021112',
       '23021045', '23018471', '23014554', '23014554',
       ...
       'USGS-15908000', 'USGS-15908000', 'USGS-15908000', 'USGS-15908000',
       'USGS-15908000', 'USGS-15905100', 'USGS-15905100', 'USGS-15905100',
       'USGS-15905100', 'USGS-15905100'],
      dtype='object', name='site_id', length=112790)

In [7]:
subs_buff = subbasins.copy()
subs_buff = subs_buff.to_crs("EPSG:6933")
subs_buff.geometry = subs_buff.buffer(2500)

hybas_pairs = gpd.sjoin(
    subs_buff,
    hydroatlas.to_crs("EPSG:6933"),
    how='left',
)

hybas_area = hybas_pairs['UP_AREA']
merit_area = hybas_pairs['uparea_km2']
hybas_pairs['hybas_area_diff'] = (hybas_area - merit_area) / ((hybas_area + merit_area)/2)

def get_best_hybas(grp):
     # Pick best match by area.
    idx_min = grp['hybas_area_diff'].abs().idxmin()
    return grp.loc[idx_min]

all_attributes = hybas_pairs.reset_index().groupby('site_id').apply(get_best_hybas, include_groups=False)

In [8]:
all_attributes

,geometry,area_km2,uparea_km2,original_comid,node_type,is_gauge,is_reservoir,nextdown,outlet_id,HYBAS_ID,...,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area,hybas_area_diff
site_id,,,,,,,,,,,,,,,,,,,,,
78007101,"POLYGON ((-11397970.627 5802126.689, -11397982...",187.878281,674.934996,78007101,original,False,False,7820071213,USGS-14246900,7.090223e+09,...,0,0,41,41815,1.465360e+05,3.770672e+05,924,1.604932,0.070765,0.360948
78007191,"POLYGON ((-11374665.52 5802166.538, -11374677....",487.056715,487.056715,78007191,original,False,False,78007101,USGS-14246900,7.090218e+09,...,0,0,41,46472,6.906823e+03,6.906823e+03,924,0.991235,0.034850,-0.597093
78007214,"POLYGON ((-11511746.018 5833993.619, -11511758...",61.349363,366.835934,78007214,split_downstream,False,False,7820071213,USGS-14246900,7.090208e+09,...,3,3,41,41007,3.155499e+06,3.155500e+06,923,1.267565,0.049119,0.002625
78007257,"POLYGON ((-11423353.344 5810576.113, -11423365...",490.765797,490.765797,78007257,original,False,False,7820071213,USGS-14246900,7.090216e+09,...,0,0,41,41792,3.898186e+05,3.898184e+05,923,1.584513,0.084067,0.253760
78007356,"POLYGON ((-11371774.079 5785242.409, -11371786...",154.550897,154.550897,78007356,original,False,False,7820071213,USGS-14246900,7.090228e+09,...,1,1,41,41007,3.531842e+04,3.531842e+04,923,1.003827,0.034359,0.513401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-14236200,"POLYGON ((-11796590.997 5323778.293, -11796603...",356.139032,356.139032,78018213,gauge,True,False,7820030793,USGS-14246900,7.090369e+09,...,56,56,240,55867,1.459517e+08,1.459517e+08,948,1.372121,0.047163,0.117755
USGS-14238000,"POLYGON ((-11827088.717 5315505.174, -11827100...",4.076160,3598.232776,78017902,gauge,True,False,78017892,USGS-14246900,7.090370e+09,...,115,30,240,55867,3.761161e+07,6.212481e+08,948,0.232961,0.002844,-0.040210
USGS-14242580,"POLYGON ((-11820960.356 5293463.373, -11820972...",431.403474,1293.566256,78017968,gauge,True,False,USGS-14243000,USGS-14246900,7.090374e+09,...,25,25,240,55867,6.733032e+06,6.733031e+06,948,1.267044,0.043682,-1.104374


In [9]:
attributes_dir = save_dir / 'attributes' 
attributes_dir.mkdir(parents=True, exist_ok=True)

In [10]:
hybas_properties = pd.read_csv(attributes_dir / "properties.txt", header=None)[0].to_list()
matchup_props = ['outlet_id', 'area_km2','uparea_km2']
all_props = matchup_props + hybas_properties + ['hybas_area_diff']

In [11]:
attributes = (
    all_attributes[all_props]
    .reset_index()
    .rename(columns={'site_id':'subbasin', 'outlet_id': 'basin'})
    .astype({'subbasin':str, 'basin': str})
    .set_index(['basin', 'subbasin'])
)
attributes

area_km2     uparea_km2   dis_m3_pyr  \
basin         subbasin                                                
USGS-14246900 78007101       187.878281     674.934996    24.731001   
              78007191       487.056715     487.056715     4.498000   
              78007214        61.349363     366.835934     9.646000   
              78007257       490.765797     490.765797    17.368000   
              78007356       154.550897     154.550897     7.621000   
...                                 ...            ...          ...   
              USGS-14236200  356.139032     356.139032    16.018999   
              USGS-14238000    4.076160    3598.232776   166.397003   
              USGS-14242580  431.403474    1293.566256    18.171000   
              USGS-14243000  470.317506    5767.696618   188.699005   
              USGS-14246900  375.176233  651408.057898  7525.879883   

                              dis_m3_pmn    dis_m3_pmx  run_mm_syr  \
basin         subbasin                                               
USGS-14246900 78007101          2.059000     78.717003         946   
              78007191          0.335000     18.511999         656   
              78007214          1.192000     29.046000         813   
              78007257          1.735000     53.702999         848   
              78007356          0.555000     26.520000        1011   
...                                  ...           ...         ...   
              USGS-14236200     3.701000     27.525000        1316   
              USGS-14238000    47.576000    288.132996        1192   
              USGS-14242580     4.211000     30.427999        1536   
              USGS-14243000    54.316002    325.016998        1152   
              USGS-14246900  3936.400879  13796.940430        1309   

                             inu_pc_smn  inu_pc_smx  inu_pc_slt  lka_pc_sse  \
basin         subbasin                                                        
USGS-14246900 78007101                6           6           8          64   
              78007191                3           4           6          33   
              78007214                3           3           3           0   
              78007257                0           0           0           3   
              78007356                5           5          11          17   
...                                 ...         ...         ...         ...   
              USGS-14236200           0           0           1           2   
              USGS-14238000           0           5          16         176   
              USGS-14242580           0           1           2          32   
              USGS-14243000           8          14          21           0   
              USGS-14246900          13          21          24           0   

                             ...  urb_pc_sse  nli_ix_sav  rdd_mk_sav  \
basin         subbasin       ...                                       
USGS-14246900 78007101       ...           0           0           0   
              78007191       ...           0           0           0   
              78007214       ...           0           8          13   
              78007257       ...           0           0           0   
              78007356       ...           0           0           0   
...                          ...         ...         ...         ...   
              USGS-14236200  ...           0         392         111   
              USGS-14238000  ...           0         741         268   
              USGS-14242580  ...           0          96         141   
              USGS-14243000  ...           0         951         416   
              USGS-14246900  ...           9        1669         735   

                             hft_ix_s93  hft_ix_s09  gad_id_smj  gdp_ud_sav  \
basin         subbasin                                                        
USGS-14246900 78007101                0           0          41       418

In [12]:
attributes.to_parquet(attributes_dir / f"static_4km2.parquet")

In [36]:
test = pd.read_parquet(attributes_dir / "static.parquet")

In [37]:
test

area_km2   uparea_km2  dis_m3_pyr  dis_m3_pmn  \
basin         subbasin                                                         
EAUF-V7200010 21000002       468.102025  4364.328752   66.342003   22.684999   
              21000020       401.905725  3625.201976   48.854000   16.743000   
              21000022       292.309508  3076.492166   28.339001    9.780000   
              21000026       481.039377  1767.807919    6.031000    2.077000   
              21000032       390.920772   621.770285    8.653000    3.024000   
...                                 ...          ...         ...         ...   
USGS-50129254 USGS-50129254   42.674493    42.674493    4.198000    1.576000   
USGS-50138000 USGS-50136400   51.402873    51.402873    0.185000    0.068000   
              USGS-50138000  261.453879   312.856752    5.788000    2.068000   
USGS-50144000 USGS-50144000  349.557308   349.557308    9.208000    3.183000   
USGS-50147800 USGS-50147800  195.872993   195.872993    5.115000    1.855000   

                             dis_m3_pmx  run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin         subbasin                                                        
EAUF-V7200010 21000002       108.996002         525           0           0   
              21000020        80.602997         537           0           0   
              21000022        46.625000         504           0           0   
              21000026         9.826000         422           0           0   
              21000032        14.126000         437           0           0   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    8.597000         578          61          69   
USGS-50138000 USGS-50136400    0.359000         598          63          79   
              USGS-50138000   11.577000         593          30          48   
USGS-50144000 USGS-50144000   19.839001         609          36          61   
USGS-50147800 USGS-50147800   10.214000         601          26          38   

                             inu_pc_slt  lka_pc_sse  ...  urb_pc_sse  \
basin         subbasin                               ...               
EAUF-V7200010 21000002                1           0  ...           0   
              21000020                0           0  ...           0   
              21000022                0           1  ...           0   
              21000026                0           0  ...           0   
              21000032                0           0  ...           0   
...                                 ...         ...  ...         ...   
USGS-50129254 USGS-50129254          72           1  ...          22   
USGS-50138000 USGS-50136400          80           0  ...          48   
              USGS-50138000          51           0  ...          27   
USGS-50144000 USGS-50144000          62           1  ...           4   
USGS-50147800 USGS-50147800          38           0  ...          39   

                             nli_ix_sav  rdd_mk_sav  hft_ix_s93  hft_ix_s09  \
basin         subbasin                                                        
EAUF-V7200010 21000002              844         711         135         106   
              21000020             1754         786         187         163   
              21000022              842         631         133          88   
              21000026              728         759         142          92   
              21000032              824         536         127          86   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254        3209        2365         296         277   
USGS-50138000 USGS-50136400        4778        5447         363         346   
              USGS-50138000        3680        2432         291         288   
USGS-50144000 USGS-50144000        1686        1292         222         173   
USGS-50147800 USGS-50147800        3856        1703         301         286  